In [ ]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import CRUD module
from crud_module import AnimalShelter

###########################
# Data Manipulation / Model
###########################

# Instantiate an instance of the class
# Username and password exist inside the "__init__" method for the crud_module 
shelter = AnimalShelter()

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(shelter.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invalid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will return a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

app = JupyterDash('RescueDogDashboard')

#########################
# Dashboard Layout / View
#########################

# Loads in the Grazioso Salvare logo image
image_filename = "Grazioso Salvare Logo.png" 
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
    html.H1("Search and Rescue Animal Dashboard", style={'textAlign': 'center'}),
    
    # Adds the logo to the top of the page and inserts a URL anchor tag to the image
    html.A(
        html.Img(
            src='data:image/png;base64,{}'.format(encoded_image.decode()),
            style={'height': '110px', 'display': 'block', 'margin': '0 auto'},
            title="Visit SNHU" # This text is visible if the user hovers over the logo
        ),
        href="https://www.snhu.edu",
        target="_blank" # Opens the link in a new tab
    ),
    
    # Unique identifier
    html.H4("Created by Bethany Adamson", style={'textAlign': 'center'}),

    # The drop down menu for filtering is displayed on the left side of the page
    html.Div([
        html.Label("Select rescue type:", style={'fontWeight': 'bold', 'textAlign': 'left', 'display': 'block'}),
        dcc.Dropdown(
            id='filter-dropdown',
            options=[
                {'label': 'All', 'value': 'All'},
                {'label': 'Water Rescue', 'value': 'Water Rescue'},
                {'label': 'Mountain/Wilderness Rescue', 'value': 'Mountain/Wilderness Rescue'},
                {'label': 'Disaster Rescue/Individual Tracking', 'value': 'Disaster Rescue/Individual Tracking'},
            ],
            value='All',
            clearable=False,
            style={'width': '50%'}
        )
    ], style={'textAlign': 'center', 'paddingBottom': 20}),

    # Data Table Section
    html.Div([
        dash_table.DataTable(
            id='datatable-id',
            columns=[
            {"name": "Animal ID", "id": "animal_id", "deletable": False, "selectable": True},
            {"name": "Breed", "id": "breed", "deletable": False, "selectable": True},
            {"name": "Name", "id": "name", "deletable": False, "selectable": True},
            {"name": "Age in Weeks", "id": "age_upon_outcome_in_weeks", "deletable": False, "selectable": True},
            {"name": "Sex", "id": "sex_upon_outcome", "deletable": False, "selectable": True},
            {"name": "Latitude", "id": "location_lat", "deletable": False, "selectable": True},
            {"name": "Longitude", "id": "location_long", "deletable": False, "selectable": True},
            {"name": "Outcome Type", "id": "outcome_type", "deletable": False, "selectable": True},        
            ],
            data=[],
            editable=False,
            row_selectable='single',
            selected_rows=[],
            sort_action="native",
            filter_action="native",
            page_action="native",
            page_current=0,
            page_size=20,
            style_table={'overflowX': 'auto'},
            style_header={
                'backgroundColor': 'rgb(230,230,230)',
                'fontWeight': 'bold'
            }
        )
    ], className='row'),

    html.Br(),
    
    # The geolocation chart and pie chart are displayed side by side
    html.Div([
        # The geolocation chart is on the left
        html.Div(
            id='map-id',
            style={'flex': '1', 'height': '500px', 'marginRight': '10px'}
        ),
        # The pie chart is on the right
        html.Div(
            dcc.Graph(id='pie-chart'),
            style={'flex': '1.5', 'height': '600px'}
        )
    ],
    style={
        'display': 'flex',
        'flexDirection': 'row',
        'justifyContent': 'center',
        'alignItems': 'stretch',
        'padding': '20px'
    }),

    # Logo with hyperlink displayed at the bottom of the page
    html.A(
        html.Img(
            src='data:image/png;base64,{}'.format(encoded_image.decode()),
            style={'height': '110px', 'display': 'block', 'margin': '0 auto'},
            title="Visit SNHU"
        ),
        href="https://www.snhu.edu",
        target="_blank"
    ),
    
    # Unique identifier
    html.H4("Created by Bethany Adamson", style={'textAlign': 'center'})
])

#####################
# Callback Section
#####################

# Update DataTable based on selected dropdown filter
@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-dropdown', 'value')]
)
def update_table(filter_value):
    # Searches the database in MongoDB based on selected filter. Utilizes the read method from the CRUD module.
    try:
        if filter_value == 'All':
            # Grazioso Salvare only needs dogs for their search and rescue operations.
            data = list(shelter.read({"animal_type": "Dog"}))
        else:
            # Filters are related to the type of rescue dog Grazioso Salvare desires
            if filter_value == 'Water Rescue':
                query = {
                    "animal_type": "Dog",
                    "breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]},
                    "sex_upon_outcome": "Intact Female",
                    "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156} 
                }

            elif filter_value == 'Mountain/Wilderness Rescue':
                query = {
                    "animal_type": "Dog",
                    "breed": {"$in": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"]},
                    "sex_upon_outcome": "Intact Male",
                    "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
                }

            elif filter_value == 'Disaster Rescue/Individual Tracking':
                query = {
                    "animal_type": "Dog",
                    "breed": {"$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound", "Rottweiler"]},
                    "sex_upon_outcome": "Intact Male",
                    "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
                }
            else:
                query = {}

            data = list(shelter.read(query))

        df = pd.DataFrame(data)
        if not df.empty:
            df = df[['animal_id', 'breed', 'name', 'age_upon_outcome_in_weeks', 'sex_upon_outcome',
                     'location_lat', 'location_long', 'outcome_type']]
            return df.to_dict('records')
        else:
            return []
        
    except Exception as e:
        print("An error occurred while attempting to load the data:", e)
        return []

# Update Leaflet Geolocation Chart when a row is selected
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, selected_row):
    if not selected_row:
        # Default view is centered on Austin, Texas
        return [
            dl.Map(center=[30.75, -97.48], zoom=6, children=[
                dl.TileLayer()
            ])
        ]
    # When a row is selected the view is centered on that location, or Austin, Texas if no coordinates exist
    else:
        dff = viewData[selected_row[0]]
        lat = dff.get('location_lat', 30.75)
        lon = dff.get('location_long', -97.48)
        
        # Return the selected rows geolocation
        return [
            dl.Map(center=[lat, lon], zoom=10, children=[
                dl.TileLayer(),
                dl.Marker(position=[lat, lon], children=[
                    dl.Tooltip(dff['breed']),
                    dl.Popup([
                        html.H4(f"Breed: {dff['breed']}"),
                        html.P(f"Age in Weeks: {dff['age_upon_outcome_in_weeks']}"),
                        html.P(f"Sex: {dff['sex_upon_outcome']}"),
                        html.P(f"Name: {dff['name']}")
                    ])
                ])
            ])
        ]

# Update pie chart based on selected dropdown filter
@app.callback(
    Output('pie-chart', 'figure'),
    [Input('filter-dropdown', 'value')]
)
def update_pie_chart(filter_value):
    # Only breeds of interest are displayed in the pie chart when 'All' is selected
    if filter_value == 'All':
        query = {"breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland", 
                                   "German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky",
                                   "Doberman Pinscher", "Golden Retriever", "Bloodhound", "Rottweiler"]}}
    elif filter_value == 'Water Rescue':
        query = {"breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]}}
    elif filter_value == 'Mountain/Wilderness Rescue':
        query = {"breed": {"$in": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"]}}
    elif filter_value == 'Disaster Rescue/Individual Tracking':
        query = {"breed": {"$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound", "Rottweiler"]}}
    else:
        query = {}

    data = list(shelter.read(query))
    df = pd.DataFrame(data)

    if df.empty or 'breed' not in df:
        return px.pie(values=[1], names=['No Data'], title="No Data Available")

    breed_counts = df['breed'].value_counts().reset_index()
    breed_counts.columns = ['breed', 'count']
    fig = px.pie(breed_counts, values='count', names='breed', title=f"{filter_value} Breeds Distribution", height=600, width=600)
    fig.update_traces(textposition='inside', textinfo='percent+label')
    return fig

# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server(mode="inline")